In [ ]:
# Kaggle & Local Environment Setup
import os
import numpy as np
import pandas as pd

# Check for Kaggle input vs local directory
kaggle_input = '/kaggle/input'
if os.path.exists(kaggle_input):
    print("Listing files in /kaggle/input:")
    for dirname, _, filenames in os.walk(kaggle_input):
        for filename in filenames:
            print(os.path.join(dirname, filename))
else:
    print("Running locally. /kaggle/input directory not found.")


In [ ]:
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
# Set path to dataset with automatic fallback (Kaggle or local)
kaggle_path = '/kaggle/input/tbi-detection-v2/tbi-detection/tbi-dataset'
local_path = 'tbi-dataset'

if os.path.exists(kaggle_path):
    dataset_path = kaggle_path
elif os.path.exists(local_path):
    dataset_path = local_path
else:
    dataset_path = 'tbi-dataset'
    print(f"Warning: Dataset path '{dataset_path}' not found. Please ensure images are placed under '{dataset_path}/<class_name>/'")

print(f"Using Dataset Path: {dataset_path}")

# Image data generator with data augmentation and validation split
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

if os.path.exists(dataset_path):
    train_data = train_datagen.flow_from_directory(
        dataset_path,
        target_size=(128, 128),
        color_mode='grayscale',
        batch_size=8,
        class_mode='categorical',
        subset='training',
        shuffle=True
    )

    val_data = val_datagen.flow_from_directory(
        dataset_path,
        target_size=(128, 128),
        color_mode='grayscale',
        batch_size=8,
        class_mode='categorical',
        subset='validation',
        shuffle=False
    )
else:
    print("Dataset path does not exist. Skipping flow_from_directory.")


In [ ]:
model = Sequential([
    Input(shape=(128, 128, 1)),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')  # 4 classes: Epidural, Intraparenchymal, Subarachnoid, Subdural
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()


In [ ]:
if 'train_data' in globals() and train_data is not None:
    history = model.fit(train_data, epochs=10, validation_data=val_data)
else:
    print("train_data is not available. Please provide dataset to train the model.")


In [ ]:
if 'history' in globals() and history is not None:
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Accuracy over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Loss over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()
else:
    print("History log not found. Skipping plots.")


In [ ]:
if 'val_data' in globals() and val_data is not None:
    val_data.reset()
    preds = model.predict(val_data)
    y_pred = np.argmax(preds, axis=1)
    y_true = val_data.classes

    # Classification Report
    print("
Classification Report:
")
    print(classification_report(y_true, y_pred, target_names=list(val_data.class_indices.keys())))

    # Normalized Confusion Matrix
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    sns.heatmap(cm * 100, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=val_data.class_indices.keys(),
                yticklabels=val_data.class_indices.keys())
    plt.title('Normalized Confusion Matrix (%)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
else:
    print("val_data is not available. Skipping evaluation.")


In [ ]:
def predict_single_image(img_path, model_obj=model, class_labels=None):
    if class_labels is None:
        if 'val_data' in globals() and hasattr(val_data, 'class_indices'):
            class_labels = list(val_data.class_indices.keys())
        else:
            class_labels = ["Epidural", "Intraparenchymal", "Subarachnoid", "Subdural"]

    if not os.path.exists(img_path):
        print(f"Error: Image file '{img_path}' not found.")
        return None

    img = image.load_img(img_path, target_size=(128, 128), color_mode='grayscale')
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model_obj.predict(img_array, verbose=0)
    class_idx = np.argmax(prediction, axis=1)[0]
    predicted_class = class_labels[class_idx]

    # Simulated severity parameters
    midline_shift = round(random.uniform(0, 10), 1)
    hematoma_volume = random.randint(10, 100)
    intracranial_pressure = random.randint(5, 35)

    if midline_shift > 5 or hematoma_volume > 40 or intracranial_pressure > 20:
        severity = "Critical"
    elif midline_shift > 2 or hematoma_volume > 20 or intracranial_pressure > 15:
        severity = "Moderate"
    else:
        severity = "Normal"

    ICH_classes = ["Epidural", "Subdural", "Subarachnoid", "Intraparenchymal"]
    ICH_Detected = int(predicted_class in ICH_classes)

    output = {
        "Predicted_Class": predicted_class,
        "ICH_Detected": ICH_Detected,
        "Epidural_Hemorrhage": int(predicted_class == "Epidural"),
        "Subdural_Hemorrhage": int(predicted_class == "Subdural"),
        "Subarachnoid_Hemorrhage": int(predicted_class == "Subarachnoid"),
        "Intraparenchymal_Hemorrhage": int(predicted_class == "Intraparenchymal"),
        "Midline_Shift": f"{midline_shift} mm",
        "Hematoma_Volume": f"{hematoma_volume} ml",
        "Intracranial_Pressure": f"{intracranial_pressure} mmHg",
        "Severity": severity
    }

    print(json.dumps(output, indent=2))
    return output


In [ ]:
# Resolve path to sample image dynamically
sample_path = 'tbi-dataset/Subarachnoid/Subarachnoid_0.png'
if not os.path.exists(sample_path):
    sample_path = '/kaggle/input/tbi-detection-v2/tbi-detection/tbi-dataset/Subarachnoid/Subarachnoid_0.png'

if os.path.exists(sample_path):
    predict_single_image(sample_path)
else:
    print(f"Sample image '{sample_path}' not found. Pass a valid image path to predict_single_image(path).")
